In [ ]:
import os

import math
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, utils

from tqdm import tqdm
from PIL import Image
from glob import glob
import importlib

import ddpm_model as ddpm
importlib.reload(ddpm)
ContextUnet = ddpm.ContextUnet

In [ ]:
class Config():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    n_feat = 128
    n_cfeat = 5
    hegiht = 16

    timesteps = 500
    start = 1e-4
    end = 2e-2
    betas = torch.linspace(start, end, timesteps).to(device=device)
    alphas = torch.tensor(1.0 - betas, device=device)
    alpha_bars = torch.cumsum(alphas.log(), dim=0).exp()
    alpha_bars[0] = 1

    batch_size = 100
    n_epoch = 300
    lr = 1e-3

    image_size=64
    channels=3

    save_dir_weight = "ddpm_weight"
    save_dir_img = "ddpm_img"
    data_path = "datas/tinyhero"

    os.makedirs(save_dir_weight, exist_ok=True)
    os.makedirs(save_dir_img, exist_ok=True)

config = Config()

/tmp/ipykernel_140168/1780614525.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  alphas = torch.tensor(1.0 - betas, device=device)


In [9]:
def q_sample(x, t, noise):
    sqrt_ab = torch.sqrt(config.alpha_bars[t])[:, None, None, None]
    sqrt_mab = torch.sqrt(1 - config.alpha_bars[t])[:, None, None, None]
    return sqrt_ab*x + sqrt_mab*noise

@torch.no_grad()
def sample(model, imgs=None, n=16):
    model.eval()
    if imgs is None:
        imgs = torch.randn(n, config.channels, config.image_size, config.image_size, device=config.device)
    else:
        n = len(imgs)

    for t in reversed(range(1, config.timesteps)):
        t_batch = torch.full((n, ), t, device=config.device, dtype=torch.long)
        t_input = t_batch.float().unsqueeze(-1)
        noise_pred = model(imgs, t_input)

        beta_t = config.betas[t]
        alpha_t = config.alphas[t]
        alpha_bar_t = config.alpha_bars[t]

        if t > 1:
            noise = torch.randn_like(imgs)
        else:
            noise = torch.zeros_like(imgs)
        
        imgs = (
            1 / torch.sqrt(alpha_t) 
            * (imgs - ((1 - alpha_t) / torch.sqrt(1-alpha_bar_t)) * noise_pred)
            + torch.sqrt(beta_t) * noise
        )

    imgs = (imgs.clamp(-1, 1) + 1) / 2
    return imgs

In [10]:
class HeroDataset(Dataset):
    def __init__(self, root, transform=None):
        self.paths = sorted(glob(os.path.join(root, "*/*.png")))
        self.transform= transform
    
    def __len__(self):
        return len(self.paths)
    
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img

transform = transforms.Compose([
    transforms.Resize((config.image_size, config.image_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

dataset = HeroDataset(config.data_path, transform)
loader = DataLoader(dataset, batch_size=config.batch_size, shuffle=True,
                    num_workers=4, pin_memory=True)

In [ ]:
def train():
    model = ContextUnet(config.channels, config.n_feat, config.n_cfeat, config.hegiht)
    model.to(config.device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)

    schedule = torch.optim.lr_scheduler.StepLR(optimizer, step_size=60, gamma=0.1)

    for epoch in range(1, config.n_epoch):
        pbar = tqdm(loader, desc=f"Epoch {epoch} / {config.n_epoch}")

        for x in pbar:
            x = x.to(config.device)
            bs = x.size(0)

            t = torch.randint(1, config.timesteps, (bs, ), device=config.device).long()
            noise = torch.randn_like(x)

            x_t = q_sample(x, t, noise)
            t_input = t.float().unsqueeze(-1)
            noise_pred = model(x_t, t_input)

            loss = F.mse_loss(noise_pred, noise)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            pbar.set_postfix({"Loss": loss.item()})

        imgs = sample(model, n=16)
        utils.save_image(imgs, f"{config.save_dir_img}/ddpm_epoch_{epoch:03d}.png", nrow=4)
        if epoch == 30:
            torch.save(model.state_dict(), f"{config.save_dir_weight}/ddpm_{epoch}epoch.pth")
    
    torch.save(model.state_dict(), f"{config.save_dir_weight}/ddpm.pth")
    print("Training Complete")

In [16]:
train()

Epoch 299 / 300: 100%|██████████| 37/37 [00:10<00:00,  3.47it/s, Loss=0.0162] 


Training Complete


In [18]:
len(loader), len(dataset)

(37, 3648)